# Federer and Nadal by surface and date

Start with their charted grass matches against all opponents, not just each other.

Data: [Jeff Sackmann / Match Charting Project](https://github.com/JeffSackmann/tennis_MatchChartingProject), CC BY-NC-SA 4.0. The data-derived results use the same license; the code is MIT.

In [1]:
# Load the local tools
import sys
from pathlib import Path

from IPython.display import Markdown, display
root = Path.cwd()
if not (root / "compare.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from compare import compare, format_markdown
snapshot = root / "data/raw/1813a1309b7ed7ebf1c7e884b32bf675d00e4edf"
if not snapshot.exists():
    raise FileNotFoundError("Run the pinned download command in the README first")

## Compare pooled point counts

Sum successes and opportunities, then divide. Averaging match percentages would give a short match the same weight as a long one. Pooled rates still mix opponents, seasons, and match conditions.

In [2]:
# Compare serve and return rates
result = compare(snapshot, ["Roger Federer", "Rafael Nadal"], surface="Grass")
print(f"{'Player':18} {'Matches':>8} {'Serve pts':>10} {'Ace %':>8} {'Return won %':>13}")
for player, values in result["players"].items():
    print(f"{player:18} {values['matches']:8} {values['serve_points']:10} "
          f"{values['ace_rate']:8.1%} {values['return_points_won']:13.1%}")

Player              Matches  Serve pts    Ace %  Return won %
Roger Federer           119      11114    11.9%         37.0%
Rafael Nadal             40       4246     7.1%         35.3%


## Pick a date window

The start is included and the cutoff is excluded. Here we combine grass and clay from 2010 through 2019. These are charted-match counts, not complete careers. Retrospective charting does not mean the data was available at the time.

In [3]:
# Check sample sizes for the 2010s on grass and clay
window = compare(snapshot, ["Roger Federer", "Rafael Nadal"],
                 surface=["Grass", "Clay"], from_date="20100101",
                 before="20200101")
print(f"Charted matches: {window['sample']['metadata_matches']}")
print(f"With player stats: {window['sample']['matches_with_stats']}")
for player, values in window["players"].items():
    print(player, values["matches"], "matches,",
          values["serve_points"], "serve points")

Charted matches: 824
With player stats: 822
Roger Federer 115 matches, 10581 serve points
Rafael Nadal 132 matches, 10206 serve points


## Match-level uncertainty

Resample whole charted matches, recalculate each pooled rate, and take the middle 95% of those results. The table keeps match and point counts in the same view as each rate and interval. These selected matches are still uneven across opponents and seasons, so the intervals are a within-sample check, not a career-wide ranking.


In [4]:
# Resample charted matches, keeping each match's points together
with_intervals = compare(snapshot, ["Roger Federer", "Rafael Nadal"],
                         surface="Grass", bootstrap=2000, seed=42)
display(Markdown(format_markdown(with_intervals)))


| Rate | Roger Federer<br>119 matches, 11,114 serve pts, 12,144 return pts | Rafael Nadal<br>40 matches, 4,246 serve pts, 4,402 return pts |
| --- | --- | --- |
| Aces | 11.9% (11.0%–12.9%) | 7.1% (5.7%–8.6%) |
| Double faults | 1.6% (1.3%–1.8%) | 2.0% (1.6%–2.3%) |
| First serves in | 65.4% (64.5%–66.4%) | 69.4% (67.9%–70.8%) |
| First serves won | 79.7% (78.7%–80.9%) | 73.7% (71.7%–75.9%) |
| Second serves won | 59.7% (58.1%–61.4%) | 57.6% (54.3%–60.7%) |
| Return points won | 37.0% (35.8%–38.2%) | 35.3% (33.4%–37.4%) |
| Break points saved | 71.8% (68.3%–75.3%) | 68.1% (61.6%–73.7%) |

Parentheses show 95% match-bootstrap intervals.

## Next questions

How much do these numbers change across seasons? How uncertain are the differences? The opponents and matches aren't evenly sampled, so I'd want to look at that before getting carried away with a ranking.